In [1]:


import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("❌ GPU tidak digunakan")

CUDA available: True
GPU: NVIDIA GeForce GTX 1660


In [ ]:

# Cek data.yaml + jumlah file dataset


import yaml
import os

DATASET_PATH = "dataset"   # ganti jika folder beda

yaml_path = os.path.join(DATASET_PATH, "data.yaml")

with open(yaml_path, 'r') as file:
    data = yaml.safe_load(file)

print("📄 ISI data.yaml:\n")
print(data)

print("\n📌 Informasi Dataset:")
print("Train path  :", data['train'])
print("Val path    :", data['val'])
print("Jumlah kelas:", data['nc'])
print("Nama kelas  :", data['names'])

print("\n📊 Jumlah file per split:")

for split in ["train", "valid", "test"]:
    img_dir = os.path.join(DATASET_PATH, split, "images")
    lbl_dir = os.path.join(DATASET_PATH, split, "labels")

    if os.path.exists(img_dir):
        n_img = len([
            f for f in os.listdir(img_dir)
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        ])

        n_lbl = len([
            f for f in os.listdir(lbl_dir)
            if f.lower().endswith(".txt")
        ]) if os.path.exists(lbl_dir) else 0

        print(f"📁 {split:6s} → {n_img:4d} citra | {n_lbl:4d} label")

    else:
        print(f"📁 {split:6s} → (folder tidak ada)")

📄 ISI data.yaml:

{'train': '../train/images', 'val': '../valid/images', 'test': '../test/images', 'nc': 12, 'names': ['broken', 'foreign_matter', 'full_black', 'full_sour', 'fungus', 'good', 'immature', 'insect_severe', 'insect_slight', 'partial_black', 'partial_sour', 'withered'], 'roboflow': {'workspace': 'faruq-reybi', 'project': 'coffee-beans-dataset-2-segmentation-f7xlx-f7ttr', 'version': 2, 'license': 'CC BY 4.0', 'url': 'https://universe.roboflow.com/faruq-reybi/coffee-beans-dataset-2-segmentation-f7xlx-f7ttr/dataset/2'}}

📌 Informasi Dataset:
Train path  : ../train/images
Val path    : ../valid/images
Jumlah kelas: 12
Nama kelas  : ['broken', 'foreign_matter', 'full_black', 'full_sour', 'fungus', 'good', 'immature', 'insect_severe', 'insect_slight', 'partial_black', 'partial_sour', 'withered']

📊 Jumlah file per split:
📁 train  → 20556 citra | 20556 label
📁 valid  → 2536 citra | 2536 label
📁 test   → 1262 citra | 1262 label


EXPERIMENT 1

In [ ]:
from ultralytics import YOLO

def main():
    # Load pretrained model
    model = YOLO("yolo11n-seg.pt")

    # Training
    results = model.train(
        data="dataset/data.yaml",
        epochs=120,
        imgsz=640,
        batch=8,
        optimizer="AdamW",
        lr0=0.001,
        project="runs",
        name="coffee-seg",
        device=0  # pakai GPU, kalau CPU ganti 'cpu'
    )

if __name__ == "__main__":
    main()

Ultralytics 8.4.26  Python-3.14.3 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce GTX 1660, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=120, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=coffee-seg, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patience=100, per

EXPERIMENT 2

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n-seg.pt")

model.train(
    data="dataset/data.yaml",
    epochs=100,
    imgsz=640,
    batch=8,

    optimizer="AdamW",
    lr0=0.0008,

    patience=20,

    workers=8,
    cache="disk",

    amp=True,

    accumulate=4,

    project="runs",
    name="coffee-seg-final",

    device=0,
    verbose=True
)